# 12 双人标注、Gold与图片评测

**用途：** 生成40行盲审包、合并裁决表并验证gold门禁为何在人工完成前必须失败。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
import pandas as pd
from tests.prepare_multimodal_double_review import prepare_packets, merge_reviews

candidates = load_jsonl(PROJECT2_ROOT / "tests" / "multimodal_real_candidates.jsonl")
show_table(pd.DataFrame(candidates)["scenario"].value_counts().rename_axis("场景").reset_index(name="数量").to_dict("records"))
check_equal("真实迁移候选数量", len(candidates), 40)

temp_dir = tempfile.TemporaryDirectory()
root = Path(temp_dir.name)
count = prepare_packets(
    PROJECT2_ROOT / "tests" / "multimodal_real_annotation_template.csv",
    root / "reviewer_a.csv",
    root / "reviewer_b.csv",
    reviewer_a="reviewer-a",
    reviewer_b="reviewer-b",
)
merged = merge_reviews(root / "reviewer_a.csv", root / "reviewer_b.csv", root / "adjudication.csv")
incomplete = sum(bool(row["incomplete_fields"]) for row in merged)
show_table([
    {"盲审A行数": count, "盲审B行数": count, "裁决表行数": len(merged), "待人工补全": incomplete}
])
check_equal("A/B各生成40行", count, 40)
check_equal("合并后仍是40行", len(merged), 40)
check_equal("未人工填写前40行都不允许成为gold", incomplete, 40)
temp_dir.cleanup()

,场景,数量
0,old_plate,10
1,serial_plate,8
2,manufacturer_plate,8
3,damage,7
4,hydraulic_part,4
5,nameplate,3


[PASS] 真实迁移候选数量 | actual=40, expected=40


,盲审A行数,盲审B行数,裁决表行数,待人工补全
0,40,40,40,40


[PASS] A/B各生成40行 | actual=40, expected=40
[PASS] 合并后仍是40行 | actual=40, expected=40
[PASS] 未人工填写前40行都不允许成为gold | actual=40, expected=40


In [3]:
from tests.build_multimodal_gold import build_gold

gate_failed_as_expected = False
with tempfile.TemporaryDirectory() as directory:
    output = Path(directory) / "gold.jsonl"
    try:
        build_gold(
            PROJECT2_ROOT / "tests" / "multimodal_real_candidates.jsonl",
            PROJECT2_ROOT / "tests" / "multimodal_real_annotation_template.csv",
            output,
        )
    except ValueError as exc:
        gate_failed_as_expected = True
        print(str(exc).splitlines()[0])
check("空白人工标注不能生成gold", gate_failed_as_expected)

Gold dataset validation failed:
[PASS] 空白人工标注不能生成gold


{'检查项': '空白人工标注不能生成gold', '状态': 'PASS', '说明': ''}

In [4]:
review_tests = run_unittest(
    ["tests.test_build_multimodal_gold", "tests.test_multimodal_double_review", "tests.test_multimodal_evaluation"],
    project2_root=PROJECT2_ROOT,
)
check("标注门禁与指标13条通过", "Ran 13 tests" in review_tests.output and "OK" in review_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest tests.test_build_multimodal_gold tests.test_multimodal_double_review tests.test_multimodal_evaluation -v
test_builds_gold_only_after_all_gates_pass (tests.test_build_multimodal_gold.BuildGoldTests.test_builds_gold_only_after_all_gates_pass) ... ok
test_rejects_same_reviewer_and_missing_readable_value (tests.test_build_multimodal_gold.BuildGoldTests.test_rejects_same_reviewer_and_missing_readable_value) ... ok
test_merge_blanks_conflicting_final_value_for_adjudication (tests.test_multimodal_double_review.MultimodalDoubleReviewTests.test_merge_blanks_conflicting_final_value_for_adjudication) ... ok
test_merge_prefills_independent_agreements (tests.test_multimodal_double_review.MultimodalDoubleReviewTests.test_merge_prefills_independent_agreements) ... ok
test_prepare_packets_are_separate_and_blinded (tests.test_multimodal_double_review.MultimodalDoubleReviewTests.test_prepare_packets_are_separate_and_blinded) ... ok
test_alias

{'检查项': '标注门禁与指标13条通过', '状态': 'PASS', '说明': ''}

## 正确人工流程

1. Reviewer A和B拿到不含模型答案的独立CSV，不能互看。
2. 两人逐张核对许可、隐私、图片类型、字段可读性、真实值、损坏和是否拒识。
3. 合并脚本只自动接受一致字段；分歧进入`conflict_fields`。
4. 第三位且不同于A/B的裁决人查看原图，填写最终列和`adjudication_reason`。
5. `build_multimodal_gold.py`通过后才运行正式视觉评测。

**成功标准不是40/40 API成功。** 必须报告字段TP/FP/FN/TN、precision/recall/F1、零件号幻觉、拒识混淆矩阵、场景分层和P50/P95。

### 面试官会问

- 为什么模型预标注不能先给两位Reviewer看？
- `readable/unreadable/not_present`如何影响FP和FN？
- 错误但非空的零件号为什么同时算FP和FN？
- 迁移机械图片为什么不能代表真实挖机生产准确率？
- 如何衡量标注一致率和裁决比例？

### 参考答案

1. **为什么Reviewer不能先看模型预标注？** 先看模型答案会产生锚定偏差，两个人可能一起接受同一个模型错误，表面一致率变高但gold失真。盲审包只给原图和字段定义，模型结果在gold冻结后才用于评测。
2. **三种可读状态如何影响指标？** `readable`表示字段存在且能标真实值，模型应命中；`unreadable`表示字段可能存在但无法可靠读取，模型输出具体值属于幻觉FP；`not_present`表示图中没有该字段，模型输出同样是FP，但不应把空输出算FN。
3. **错误非空件号为何同时FP和FN？** 模型输出了一个不正确值，产生一个错误预测，所以有FP；同时正确gold值没有被预测出来，所以有FN。这能同时惩罚“编错”和“漏掉正确值”。
4. **为什么迁移图片不能代表生产准确率？** 当前40张是开放许可机械图片，品牌、拍摄设备、光照和客户操作与真实挖机售后分布不同。它们适合验证流程和发现困难场景，生产结论必须再用授权脱敏的真实业务图片分层评测。
5. **如何衡量一致率和裁决比例？** 对枚举字段可报告百分比一致率或Cohen's kappa，对文本字段先按规范化/别名规则判断一致；裁决比例=`存在conflict_fields的样本数/总样本数`，并按零件号、损坏、拒识等字段分别统计。

**代码落点：** `tests/prepare_multimodal_double_review.py`、`build_multimodal_gold.py`、`multimodal_evaluation.py`。